# File I/O (NumPy)

> Generated from [`09-File-IO.py`](09-File-IO.py) by `tools/build_notebooks.py` - edit the `.py` (or its `.md` notes), not this notebook. Changes made here are overwritten on the next build.

## Quick notes

### What this covers
Saving and loading NumPy arrays: binary `.npy`/`.npz` formats, human-readable
text/CSV output, and memory-mapped arrays for datasets too large to fit in RAM.

### Key points
- `np.save("file.npy", arr)` / `np.load("file.npy")` — single array, binary,
  exact dtype/shape/values preserved, fast and compact.
- `np.savez("file.npz", name1=arr1, name2=arr2)` — bundles multiple named
  arrays into one file; load and access via `bundle["name1"]`.
  `np.savez_compressed()` does the same with zip compression (smaller file,
  slower to read/write — worth it for large, repetitive arrays).
- `np.savetxt("file.csv", arr, delimiter=",", header="...")` /
  `np.loadtxt("file.csv", delimiter=",", skiprows=1)` — human-readable
  text, editable/viewable outside NumPy, but loses exact dtype (everything
  reloads as `float64`).
- `np.memmap("file.dat", dtype=..., mode=..., shape=...)` — array backed by
  a file on disk; NumPy reads/writes only the slices you touch instead of
  loading everything into RAM. Lets you work with arrays bigger than
  available memory, at the cost of disk-speed access instead of RAM-speed.

### Gotchas (the part worth revising twice)
- `savetxt`/`loadtxt` round-trips lose integer dtype — an int column comes
  back as `float64`. Use `.npy`/`.npz` when exact types matter.
- `savez_compressed` can produce a *larger* file than plain `savez` on tiny
  arrays — compression overhead only pays off at scale.
- `np.memmap` requires you to specify `dtype` and `shape` explicitly when
  reopening — the raw `.dat` file has no header describing its own contents.
  Get either wrong and you'll silently read garbage.
- Always `del` (or let go out of scope) a memmap, or call `.flush()`, before
  assuming writes have hit disk.

### When it matters
`.npy`/`.npz` for fast intermediate results between pipeline stages;
`savetxt`/CSV for sharing data with non-NumPy tools or humans; `memmap` for
out-of-core processing of datasets larger than RAM (common in scientific
computing and large-scale data analysis).

### One-line recap example
```python
import numpy as np
np.save("weights.npy", np.array([0.1, 0.2, 0.3]))
loaded = np.load("weights.npy")   # exact dtype/values preserved
```

In [ ]:
# 01. np.save / np.load - Single Array (.npy binary format)
# 02. np.savez / np.savez_compressed - Multiple Named Arrays (.npz)
# 03. np.savetxt / np.loadtxt - Human-Readable Text Output
# 04. np.memmap - Memory-Mapped Arrays for Files Too Large for RAM
# 05. Cleanup

import numpy as np
import os

## 01. np.save / np.load - Single Array (.npy binary format)

- np.save() writes ONE array to a binary ".npy" file - preserves dtype,
  shape, and exact values (no precision loss, unlike text formats).
- np.load() reads it back as an identical array.
- Fast and compact - the default choice for saving intermediate NumPy results.

In [ ]:
original = np.array([[1.5, 2.5, 3.5], [4.5, 5.5, 6.5]])
np.save("temp_array.npy", original)
print("Saved original array to temp_array.npy:\n", original)

loaded = np.load("temp_array.npy")
print("Loaded back:\n", loaded)
print("Arrays equal:", np.array_equal(original, loaded))
print("dtype preserved:", loaded.dtype)

## 02. np.savez / np.savez_compressed - Multiple Named Arrays (.npz)

- np.savez() bundles SEVERAL arrays into one ".npz" file, each given a name
  (like a dict). Load it back and access each array by that name.
- np.savez_compressed() does the same but applies zip compression - smaller
  file, slower save/load. Use for large arrays with lots of repeated values.

In [ ]:
weights = np.array([0.1, 0.2, 0.3])
biases = np.array([1.0, 2.0])
np.savez("temp_bundle.npz", weights=weights, biases=biases)
print("Saved weights + biases to temp_bundle.npz")

bundle = np.load("temp_bundle.npz")
print("Keys in bundle:", list(bundle.keys()))
print("bundle['weights'] =", bundle["weights"])
print("bundle['biases']  =", bundle["biases"])

np.savez_compressed("temp_bundle_compressed.npz", weights=weights, biases=biases)
uncompressed_size = os.path.getsize("temp_bundle.npz")
compressed_size = os.path.getsize("temp_bundle_compressed.npz")
print(f"\nUncompressed .npz size: {uncompressed_size} bytes")
print(f"Compressed .npz size:   {compressed_size} bytes")
print("# For small arrays like these, compression overhead can make the file LARGER -")
print("#   it pays off on big, repetitive arrays, not tiny demo ones.")

## 03. np.savetxt / np.loadtxt - Human-Readable Text Output

- np.savetxt() writes an array as plain text (CSV-like) - readable in any
  text editor or spreadsheet, but loses exact float precision and dtype
  info unless you're careful with format strings.
- delimiter= controls the column separator; header= adds a comment line.
- np.loadtxt() reads it back (always as floats, unless dtype specified).

In [ ]:
table = np.array([[1, 85.5, 90.0], [2, 72.3, 88.1], [3, 91.0, 76.4]])
np.savetxt("temp_scores.csv", table, delimiter=",", header="id,test1,test2", comments="")
print("Saved table to temp_scores.csv:\n", table)

with open("temp_scores.csv") as f:
    print("\nRaw file contents:")
    print(f.read())

reloaded_table = np.loadtxt("temp_scores.csv", delimiter=",", skiprows=1)
print("Reloaded from text (all floats now):\n", reloaded_table)
print("# Note: loadtxt has no concept of an int column - everything comes back as float64.")

## 04. np.memmap - Memory-Mapped Arrays for Files Too Large for RAM

- A memmap array is backed by a file on disk, not loaded fully into RAM -
  NumPy reads/writes only the slices you actually touch, on demand.
- Lets you work with arrays far bigger than available memory (e.g. a
  100GB dataset on an 8GB-RAM machine), at the cost of disk I/O speed
  for each access instead of RAM speed.
- mode='w+' creates/overwrites; mode='r' opens read-only; dtype and shape
  must be specified explicitly (the file has no header describing them).

In [ ]:
mm_shape = (1000, 1000)
mm = np.memmap("temp_bigarray.dat", dtype="float64", mode="w+", shape=mm_shape)
mm[0, 0] = 42.0
mm[999, 999] = -1.0
mm.flush()  # ensure writes are pushed to disk
print(f"Created memmap array, shape={mm_shape}, on-disk size ~ "
      f"{os.path.getsize('temp_bigarray.dat') / 1e6:.1f} MB")
print("mm[0, 0] =", mm[0, 0], ", mm[999, 999] =", mm[999, 999])
del mm  # close the memmap (flushes and releases the file handle)

mm_reopened = np.memmap("temp_bigarray.dat", dtype="float64", mode="r", shape=mm_shape)
print("Reopened read-only: mm_reopened[0, 0] =", mm_reopened[0, 0],
      ", mm_reopened[999, 999] =", mm_reopened[999, 999])
del mm_reopened

## 05. Cleanup

In [ ]:
for fname in ("temp_array.npy", "temp_bundle.npz", "temp_bundle_compressed.npz",
              "temp_scores.csv", "temp_bigarray.dat"):
    os.remove(fname)
print("temp files removed")

print("\n# End of File I/O Explanation")